In [1]:
import sys
from pathlib import Path

package_root = str(Path.cwd().parent)
if package_root not in sys.path:
    sys.path.append(package_root)

In [2]:
from __future__ import annotations
import pandas as pd 
import json
from typing import Dict, List, Tuple, Set
from pathlib import Path
from collections.abc import Iterable
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import time
import re
import numpy as np
from copy import deepcopy
from itertools import combinations
from datetime import datetime
from typing import Any
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import time
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [3]:
DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
INPUT_FOLDER = DATASET_FOLDER + "/input-data"
DATASET2 = "dataset2"
DEVICE_TO_BE_INCLUDE = "device_to_be_include.json"
FEATURES_FOLDER = "dataset2-feature-window"
FOLD_METADATA_FILE = "training_and_validation_fold.json"
MAC_TO_FILE_MAPPING = DEVICE_MAPPING_FOLDER + "/" + DATASET2 + "/mac_to_file_mapping.csv" 
MAC_TO_BE_INCLUDE = DEVICE_MAPPING_FOLDER + "/" + DATASET2 + "/" + DEVICE_TO_BE_INCLUDE

In [4]:
MAPPING_LIST_2 = f"{DEVICE_MAPPING_FOLDER}/{DATASET2}/list_of_devices_processed.csv"

In [5]:
device_mapping_2 = pd.read_csv(MAPPING_LIST_2)
device_dict_2 = (
    device_mapping_2
    .set_index("mac_address")
    .drop(columns=["mac_address"], errors="ignore")
    .to_dict(orient="index")
)
# device_dict_2

In [6]:
mac_to_file_mapping_raw = pd.read_csv(MAC_TO_FILE_MAPPING, sep="|")

In [7]:
mac_to_be_include = set(mac_to_file_mapping_raw["mac"].to_list())
mac_to_be_include

{'00:03:7f:ee:4f:e5',
 '0c:8c:24:7f:34:84',
 '24:6f:28:d9:a7:64',
 '24:7d:4d:9c:f2:81',
 '28:6d:97:7e:a4:25',
 '2c:aa:8e:a1:27:65',
 '30:4a:26:12:14:f1',
 '54:2b:57:29:92:a9',
 '54:2b:57:29:b4:6c',
 '58:b3:fc:68:a6:e2',
 '64:16:66:73:e6:e0',
 '6c:21:a2:90:19:b0',
 '78:b2:13:e4:a6:ec',
 '78:db:2f:db:43:48',
 '7c:25:da:2d:a4:70',
 '7c:a7:b0:dc:a0:1a',
 '8c:f7:10:a1:a5:9f',
 '90:e2:02:30:80:a8',
 '9c:3d:cf:94:aa:c3',
 'a4:cf:12:32:5b:88',
 'b0:09:da:12:90:23',
 'b8:b7:f1:2a:10:fd',
 'c4:6e:7b:0e:62:5c',
 'c4:6e:7b:41:5f:28',
 'd4:d2:d6:3b:27:51',
 'ec:b5:fa:0f:15:92',
 'f4:b8:5e:35:67:b0',
 'f4:b8:5e:cd:fe:2f',
 'f4:b8:5e:ff:2b:1b',
 'f4:cf:a2:eb:59:c4'}

In [8]:
mac_to_file_mapping = {}
for mac in device_dict_2:
    if mac in mac_to_be_include:
        mac_to_file_mapping[mac] = device_dict_2[mac]
    else:
        print(f"Mac {mac}: NOT FOUND")

# mac_to_file_mapping

Mac 18:b4:30:8f:ec:23: NOT FOUND
Mac 4c:17:44:a4:0b:30: NOT FOUND
Mac 08:84:9d:09:4a:69: NOT FOUND
Mac fc:49:2d:76:33:a5: NOT FOUND
Mac 0c:8c:24:72:71:9a: NOT FOUND
Mac 98:da:c4:71:ec:9d: NOT FOUND
Mac 00:f6:20:7a:fc:b1: NOT FOUND
Mac 3c:e1:a1:62:d8:60: NOT FOUND
Mac 30:45:11:3a:17:ed: NOT FOUND
Mac a8:10:87:2a:4e:17: NOT FOUND
Mac b8:b7:f1:05:15:39: NOT FOUND


In [9]:
len(list(mac_to_file_mapping.keys()))

30

In [10]:
with open(MAC_TO_BE_INCLUDE, "w") as f:
    json.dump(mac_to_file_mapping, f, indent=4)

In [11]:
# all_windows = [ int(folder.stem) for folder in Path(FEATURES_FOLDER).iterdir()]
random_device = [folder.stem for folder in Path(FEATURES_FOLDER).iterdir()][0]

all_windows = [int(folder.stem) for folder in Path(FEATURES_FOLDER + "/" + random_device).iterdir()]
all_windows.sort()
len(all_windows)

50

In [12]:
for window in all_windows:
    data = []
    for mac in mac_to_be_include:
        df = pd.read_csv(FEATURES_FOLDER + "/" + mac + "/" + str(window) + ".csv")
        data.append(df)

    data_df = pd.concat(data)
    window_folder = FEATURES_FOLDER + "/" + str(window)
    Path(window_folder).mkdir(parents=True, exist_ok=True)
    data_df.to_parquet(
        f"{window_folder}/data.parquet",
        index=False
    )
